# 05 Make Chronological Splits

## Purpose

Create time-aware train, validation, and test splits from the modelling datasets so that later recommendation experiments are evaluated under realistic temporal conditions.

This notebook follows the earlier preprocessing stages and prepares split outputs for:
- collaborative filtering
- matrix factorisation
- later feature engineering
- dashboard and evaluation support

## Objectives

- load the modelling-ready datasets
- confirm date ranges and required columns
- sort interactions chronologically
- build train / validation / test splits using a global chronological strategy
- fit ID encoders using train data only
- apply mappings to validation and test without leakage
- identify unseen users and items outside train
- save split datasets and mapping tables
- save summary tables for reporting

In [20]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.paths import (
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    SPLITS_DIR,
    MAPPINGS_DIR,
    ensure_directories,
)

ensure_directories()
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MAPPINGS_DIR.mkdir(parents=True, exist_ok=True)

print("Output directories are ready.")

Output directories are ready.


## 1. Load modelling datasets

Load the modelling-ready datasets produced in Phase 4.

The following processed inputs are used:
- `model_explicit.parquet`
- `model_implicit.parquet`
- `model_interaction_recipe_joined.parquet`

The joined dataset is retained for later feature engineering and inspection, while the explicit and implicit datasets are the main inputs for split creation.

In [21]:
explicit_model_df = pd.read_parquet(PROCESSED_DIR / "model_explicit.parquet")
implicit_model_df = pd.read_parquet(PROCESSED_DIR / "model_implicit.parquet")
joined_model_df = pd.read_parquet(PROCESSED_DIR / "model_interaction_recipe_joined.parquet")

print("Explicit modelling dataset shape:", explicit_model_df.shape)
print("Implicit modelling dataset shape:", implicit_model_df.shape)
print("Joined modelling dataset shape:", joined_model_df.shape)

display(explicit_model_df.head(3))
display(implicit_model_df.head(3))
display(joined_model_df.head(3))

Explicit modelling dataset shape: (1071520, 7)
Implicit modelling dataset shape: (1132367, 7)
Joined modelling dataset shape: (1132367, 24)


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback
0,2008,992,2000-01-25,5,5,1,1
1,2008,3603,2000-01-25,4,4,1,1
2,2046,517,2000-02-25,5,5,1,1


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback
0,2008,992,2000-01-25,5,1,0,1
1,2008,3603,2000-01-25,4,1,0,1
2,2046,517,2000-02-25,5,1,0,1


,user_id,recipe_id,date,rating,explicit_rating,review_exists,is_unrated_observation,implicit_feedback,name,minutes,submitted,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,calorie_level,has_pp_features
0,2008,992,2000-01-25,5,5.0,1,0,1,jalapeno pepper poppers,30,1999-09-06,11,10,23,40,12,11,10,7,168.4,24.057143,1,0,1.0
1,2008,3603,2000-01-25,4,4.0,1,0,1,hooters buffalo wings,27,1999-09-24,16,12,21,55,28,16,12,7,1257.1,179.585714,1,<NA>,NaN
2,2046,517,2000-02-25,5,5.0,1,0,1,greek stuffed meatloaf,149,1999-08-23,15,20,22,41,20,15,20,7,555.4,79.342857,1,1,1.0


## 2. Confirm required columns and date behaviour

Earlier phases already standardised and date-parsed the interaction data, and saved modelling-ready outputs for later chronological splitting.

This section confirms that the expected columns are present before splitting begins.

In [22]:
print("Explicit columns:")
print(explicit_model_df.columns.tolist())

print("\nImplicit columns:")
print(implicit_model_df.columns.tolist())

print("\nJoined columns:")
print(joined_model_df.columns.tolist())

required_explicit_cols = [
    "user_id",
    "recipe_id",
    "date",
    "explicit_rating",
]

required_implicit_cols = [
    "user_id",
    "recipe_id",
    "date",
    "implicit_feedback",
]

missing_explicit = [col for col in required_explicit_cols if col not in explicit_model_df.columns]
missing_implicit = [col for col in required_implicit_cols if col not in implicit_model_df.columns]

print("\nMissing explicit columns:", missing_explicit)
print("Missing implicit columns:", missing_implicit)

print("\nExplicit date range:", explicit_model_df["date"].min(), "to", explicit_model_df["date"].max())
print("Implicit date range:", implicit_model_df["date"].min(), "to", implicit_model_df["date"].max())
print("Joined date range:", joined_model_df["date"].min(), "to", joined_model_df["date"].max())

Explicit columns:
['user_id', 'recipe_id', 'date', 'rating', 'explicit_rating', 'review_exists', 'implicit_feedback']

Implicit columns:
['user_id', 'recipe_id', 'date', 'rating', 'review_exists', 'is_unrated_observation', 'implicit_feedback']

Joined columns:
['user_id', 'recipe_id', 'date', 'rating', 'explicit_rating', 'review_exists', 'is_unrated_observation', 'implicit_feedback', 'name', 'minutes', 'submitted', 'n_steps', 'n_ingredients', 'name_length', 'description_length', 'tag_count', 'step_count_from_list', 'ingredient_count_from_list', 'nutrition_vector_length', 'nutrition_sum', 'nutrition_mean', 'has_description', 'calorie_level', 'has_pp_features']

Missing explicit columns: []
Missing implicit columns: []

Explicit date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00
Implicit date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00
Joined date range: 2000-01-25 00:00:00 to 2018-12-20 00:00:00


## 3. Define chronological split settings

A time-aware split is used instead of a random split so that future behaviour is not leaked into model training.

Recommended proportions:
- oldest 70% -> train
- next 15% -> validation
- latest 15% -> test

This matches the preprocessing plan and is appropriate for realistic recommendation evaluation.

In [23]:
TRAIN_FRAC = 0.70
VALID_FRAC = 0.15
TEST_FRAC = 0.15

assert round(TRAIN_FRAC + VALID_FRAC + TEST_FRAC, 5) == 1.0

print("Train fraction:", TRAIN_FRAC)
print("Validation fraction:", VALID_FRAC)
print("Test fraction:", TEST_FRAC)

Train fraction: 0.7
Validation fraction: 0.15
Test fraction: 0.15


## 4. Sort datasets chronologically

Before splitting, rows are sorted by:
1. `date`
2. `user_id`
3. `recipe_id`

This ensures deterministic ordering for reproducibility and supports later time-aware modelling.

In [24]:
explicit_model_df = explicit_model_df.sort_values(
    by=["date", "user_id", "recipe_id"]
).reset_index(drop=True)

implicit_model_df = implicit_model_df.sort_values(
    by=["date", "user_id", "recipe_id"]
).reset_index(drop=True)

joined_model_df = joined_model_df.sort_values(
    by=["date", "user_id", "recipe_id"]
).reset_index(drop=True)

print("Chronological sorting complete.")
display(explicit_model_df.head(3))
display(implicit_model_df.head(3))
display(joined_model_df.head(3))

Chronological sorting complete.


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback
0,2008,992,2000-01-25,5,5,1,1
1,2008,3603,2000-01-25,4,4,1,1
2,2046,517,2000-02-25,5,5,1,1


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback
0,2008,992,2000-01-25,5,1,0,1
1,2008,3603,2000-01-25,4,1,0,1
2,2046,517,2000-02-25,5,1,0,1


,user_id,recipe_id,date,rating,explicit_rating,review_exists,is_unrated_observation,implicit_feedback,name,minutes,submitted,n_steps,n_ingredients,name_length,description_length,tag_count,step_count_from_list,ingredient_count_from_list,nutrition_vector_length,nutrition_sum,nutrition_mean,has_description,calorie_level,has_pp_features
0,2008,992,2000-01-25,5,5.0,1,0,1,jalapeno pepper poppers,30,1999-09-06,11,10,23,40,12,11,10,7,168.4,24.057143,1,0,1.0
1,2008,3603,2000-01-25,4,4.0,1,0,1,hooters buffalo wings,27,1999-09-24,16,12,21,55,28,16,12,7,1257.1,179.585714,1,<NA>,NaN
2,2046,517,2000-02-25,5,5.0,1,0,1,greek stuffed meatloaf,149,1999-08-23,15,20,22,41,20,15,20,7,555.4,79.342857,1,1,1.0


## 5. Define reusable helper functions

The notebook uses:
- a global chronological splitter
- a train-only encoder builder
- a mapping applier for validation and test
- a compact split summary helper

In [25]:
def chronological_split(
    df: pd.DataFrame,
    train_frac: float = 0.70,
    valid_frac: float = 0.15,
    test_frac: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split a dataframe into train, validation, and test partitions using
    chronological row order.

    The dataframe is assumed to already be sorted by date and stable tie-breakers.
    """
    if len(df) == 0:
        return df.copy(), df.copy(), df.copy()

    n = len(df)
    train_end = int(np.floor(n * train_frac))
    valid_end = int(np.floor(n * (train_frac + valid_frac)))

    train_df = df.iloc[:train_end].copy()
    valid_df = df.iloc[train_end:valid_end].copy()
    test_df = df.iloc[valid_end:].copy()

    return train_df, valid_df, test_df


def build_id_maps_from_train(train_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build user and recipe index mappings using train data only.
    """
    user_map = (
        pd.DataFrame({"user_id": sorted(train_df["user_id"].dropna().unique())})
        .reset_index()
        .rename(columns={"index": "user_idx"})
    )
    user_map["user_idx"] = user_map["user_idx"].astype("int64")
    user_map["user_id"] = user_map["user_id"].astype("int64")

    recipe_map = (
        pd.DataFrame({"recipe_id": sorted(train_df["recipe_id"].dropna().unique())})
        .reset_index()
        .rename(columns={"index": "item_idx"})
    )
    recipe_map["item_idx"] = recipe_map["item_idx"].astype("int64")
    recipe_map["recipe_id"] = recipe_map["recipe_id"].astype("int64")

    return user_map, recipe_map


def apply_id_maps(
    df: pd.DataFrame,
    user_map: pd.DataFrame,
    recipe_map: pd.DataFrame,
) -> pd.DataFrame:
    """
    Apply train-fitted mappings to a dataframe.

    Unseen users or recipes remain missing in the mapped index columns.
    """
    out = df.merge(user_map, on="user_id", how="left")
    out = out.merge(recipe_map, on="recipe_id", how="left")
    return out


def split_summary(df: pd.DataFrame, split_name: str) -> dict:
    """
    Create a compact split summary dictionary.
    """
    return {
        "split": split_name,
        "rows": int(len(df)),
        "users": int(df["user_id"].nunique()) if "user_id" in df.columns else 0,
        "recipes": int(df["recipe_id"].nunique()) if "recipe_id" in df.columns else 0,
        "min_date": str(df["date"].min()) if len(df) > 0 else None,
        "max_date": str(df["date"].max()) if len(df) > 0 else None,
    }

## 6. Create chronological splits for the explicit modelling dataset

The explicit dataset is used for rating-aware experiments and should only contain valid explicit ratings, as established in Phase 4.

In [26]:
explicit_train, explicit_valid, explicit_test = chronological_split(
    explicit_model_df,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    test_frac=TEST_FRAC,
)

print("Explicit train shape:", explicit_train.shape)
print("Explicit valid shape:", explicit_valid.shape)
print("Explicit test shape:", explicit_test.shape)

display(pd.DataFrame([
    split_summary(explicit_train, "explicit_train"),
    split_summary(explicit_valid, "explicit_valid"),
    split_summary(explicit_test, "explicit_test"),
]))

Explicit train shape: (750064, 7)
Explicit valid shape: (160728, 7)
Explicit test shape: (160728, 7)


,split,rows,users,recipes,min_date,max_date
0,explicit_train,750064,97451,179915,2000-01-25 00:00:00,2010-04-24 00:00:00
1,explicit_valid,160728,37816,76045,2010-04-24 00:00:00,2012-09-09 00:00:00
2,explicit_test,160728,82968,61206,2012-09-09 00:00:00,2018-12-20 00:00:00


## 7. Create chronological splits for the implicit modelling dataset

The implicit dataset retains all observed interactions, including unrated observations where `rating = 0`, because those rows were previously defined as observed but unrated interactions rather than explicit negative feedback.

In [27]:
implicit_train, implicit_valid, implicit_test = chronological_split(
    implicit_model_df,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    test_frac=TEST_FRAC,
)

print("Implicit train shape:", implicit_train.shape)
print("Implicit valid shape:", implicit_valid.shape)
print("Implicit test shape:", implicit_test.shape)

display(pd.DataFrame([
    split_summary(implicit_train, "implicit_train"),
    split_summary(implicit_valid, "implicit_valid"),
    split_summary(implicit_test, "implicit_test"),
]))

Implicit train shape: (792656, 7)
Implicit valid shape: (169855, 7)
Implicit test shape: (169856, 7)


,split,rows,users,recipes,min_date,max_date
0,implicit_train,792656,106304,186564,2000-01-25 00:00:00,2010-06-17 00:00:00
1,implicit_valid,169855,46047,78532,2010-06-17 00:00:00,2012-12-28 00:00:00
2,implicit_test,169856,97133,61095,2012-12-28 00:00:00,2018-12-20 00:00:00


## 8. Create chronological splits for the joined interaction-recipe dataset

The joined dataset is split separately so that later feature engineering and dashboard support can use temporally aligned partitions if needed.

In [28]:
joined_train, joined_valid, joined_test = chronological_split(
    joined_model_df,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    test_frac=TEST_FRAC,
)

print("Joined train shape:", joined_train.shape)
print("Joined valid shape:", joined_valid.shape)
print("Joined test shape:", joined_test.shape)

display(pd.DataFrame([
    split_summary(joined_train, "joined_train"),
    split_summary(joined_valid, "joined_valid"),
    split_summary(joined_test, "joined_test"),
]))

Joined train shape: (792656, 24)
Joined valid shape: (169855, 24)
Joined test shape: (169856, 24)


,split,rows,users,recipes,min_date,max_date
0,joined_train,792656,106304,186564,2000-01-25 00:00:00,2010-06-17 00:00:00
1,joined_valid,169855,46047,78532,2010-06-17 00:00:00,2012-12-28 00:00:00
2,joined_test,169856,97133,61095,2012-12-28 00:00:00,2018-12-20 00:00:00


## 9. Fit user and item encoders on train only

To avoid leakage, ID mappings must be learned only from the training partition.

Separate mappings are created for:
- the explicit dataset
- the implicit dataset

This keeps each modelling setup self-contained.

In [29]:
explicit_user_map, explicit_recipe_map = build_id_maps_from_train(explicit_train)
implicit_user_map, implicit_recipe_map = build_id_maps_from_train(implicit_train)

print("Explicit train user map size:", explicit_user_map.shape)
print("Explicit train recipe map size:", explicit_recipe_map.shape)

print("Implicit train user map size:", implicit_user_map.shape)
print("Implicit train recipe map size:", implicit_recipe_map.shape)

display(explicit_user_map.head(3))
display(explicit_recipe_map.head(3))
display(implicit_user_map.head(3))
display(implicit_recipe_map.head(3))

Explicit train user map size: (97451, 2)
Explicit train recipe map size: (179915, 2)
Implicit train user map size: (106304, 2)
Implicit train recipe map size: (186564, 2)


,user_idx,user_id
0,0,1533
1,1,1535
2,2,1634


,item_idx,recipe_id
0,0,38
1,1,39
2,2,40


,user_idx,user_id
0,0,1533
1,1,1535
2,2,1634


,item_idx,recipe_id
0,0,38
1,1,39
2,2,40


In [30]:
explicit_user_map, explicit_recipe_map = build_id_maps_from_train(explicit_train)
implicit_user_map, implicit_recipe_map = build_id_maps_from_train(implicit_train)

print("Explicit train user map size:", explicit_user_map.shape)
print("Explicit train recipe map size:", explicit_recipe_map.shape)

print("Implicit train user map size:", implicit_user_map.shape)
print("Implicit train recipe map size:", implicit_recipe_map.shape)

display(explicit_user_map.head(3))
display(explicit_recipe_map.head(3))
display(implicit_user_map.head(3))
display(implicit_recipe_map.head(3))

Explicit train user map size: (97451, 2)
Explicit train recipe map size: (179915, 2)
Implicit train user map size: (106304, 2)
Implicit train recipe map size: (186564, 2)


,user_idx,user_id
0,0,1533
1,1,1535
2,2,1634


,item_idx,recipe_id
0,0,38
1,1,39
2,2,40


,user_idx,user_id
0,0,1533
1,1,1535
2,2,1634


,item_idx,recipe_id
0,0,38
1,1,39
2,2,40


## 10. Apply train-fitted mappings to all explicit splits

Rows in validation or test containing unseen users or recipes will produce missing mapped indices.

This behaviour is useful because it reveals cold-start exposure outside the training catalogue.

In [31]:
explicit_train_mapped = apply_id_maps(explicit_train, explicit_user_map, explicit_recipe_map)
explicit_valid_mapped = apply_id_maps(explicit_valid, explicit_user_map, explicit_recipe_map)
explicit_test_mapped = apply_id_maps(explicit_test, explicit_user_map, explicit_recipe_map)

print("Explicit mapped splits created.")
display(explicit_train_mapped.head(3))
display(explicit_valid_mapped.head(3))
display(explicit_test_mapped.head(3))

Explicit mapped splits created.


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback,user_idx,item_idx
0,2008,992,2000-01-25,5,5,1,1,10,391
1,2008,3603,2000-01-25,4,4,1,1,10,1107
2,2046,517,2000-02-25,5,5,1,1,12,234


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback,user_idx,item_idx
0,1127455,17051,2010-04-24,5,5,1,1,80438.0,7733.0
1,1157312,378398,2010-04-24,5,5,1,1,81659.0,171619.0
2,1182971,110139,2010-04-24,5,5,1,1,82741.0,61135.0


,user_id,recipe_id,date,rating,explicit_rating,review_exists,implicit_feedback,user_idx,item_idx
0,1279986,36975,2012-09-09,5,5,1,1,NaN,19463.0
1,1328942,87047,2012-09-09,5,5,1,1,88409.0,47768.0
2,1355934,193454,2012-09-09,5,5,1,1,89474.0,102528.0


## 11. Apply train-fitted mappings to all implicit splits

In [32]:
implicit_train_mapped = apply_id_maps(implicit_train, implicit_user_map, implicit_recipe_map)
implicit_valid_mapped = apply_id_maps(implicit_valid, implicit_user_map, implicit_recipe_map)
implicit_test_mapped = apply_id_maps(implicit_test, implicit_user_map, implicit_recipe_map)

print("Implicit mapped splits created.")
display(implicit_train_mapped.head(3))
display(implicit_valid_mapped.head(3))
display(implicit_test_mapped.head(3))

Implicit mapped splits created.


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback,user_idx,item_idx
0,2008,992,2000-01-25,5,1,0,1,11,410
1,2008,3603,2000-01-25,4,1,0,1,11,1154
2,2046,517,2000-02-25,5,1,0,1,13,250


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback,user_idx,item_idx
0,242205,55768,2010-06-17,5,1,0,1,27992.0,30765.0
1,242729,391308,2010-06-17,5,1,0,1,28058.0,NaN
2,254446,223104,2010-06-17,5,1,0,1,29303.0,118077.0


,user_id,recipe_id,date,rating,review_exists,is_unrated_observation,implicit_feedback,user_idx,item_idx
0,141284,15646,2012-12-28,5,1,0,1,16396.0,7048.0
1,155896,41728,2012-12-28,4,1,0,1,18012.0,22676.0
2,158966,121583,2012-12-28,5,1,0,1,18380.0,68592.0


## 12. Measure unseen users and items outside train

Validation and test may contain users or recipes that were not observed in train.

These rows are not necessarily errors, but they should be quantified because:
- collaborative filtering cannot score completely unseen users/items without fallback logic
- cold-start behaviour should be documented
- later modelling decisions may filter or separately evaluate these cases

In [33]:
def unseen_mapping_summary(df: pd.DataFrame, split_name: str) -> dict:
    return {
        "split": split_name,
        "rows": int(len(df)),
        "rows_missing_user_idx": int(df["user_idx"].isna().sum()),
        "rows_missing_item_idx": int(df["item_idx"].isna().sum()),
        "rows_missing_either_idx": int((df["user_idx"].isna() | df["item_idx"].isna()).sum()),
        "pct_missing_either_idx": round(
            ((df["user_idx"].isna() | df["item_idx"].isna()).mean() * 100), 4
        ) if len(df) > 0 else 0.0,
    }

explicit_unseen_summary = pd.DataFrame([
    unseen_mapping_summary(explicit_valid_mapped, "explicit_valid"),
    unseen_mapping_summary(explicit_test_mapped, "explicit_test"),
])

implicit_unseen_summary = pd.DataFrame([
    unseen_mapping_summary(implicit_valid_mapped, "implicit_valid"),
    unseen_mapping_summary(implicit_test_mapped, "implicit_test"),
])

print("Explicit unseen summary:")
display(explicit_unseen_summary)

print("Implicit unseen summary:")
display(implicit_unseen_summary)

Explicit unseen summary:


,split,rows,rows_missing_user_idx,rows_missing_item_idx,rows_missing_either_idx,pct_missing_either_idx
0,explicit_valid,160728,39021,44729,76352,47.5039
1,explicit_test,160728,101060,44705,122458,76.1896


Implicit unseen summary:


,split,rows,rows_missing_user_idx,rows_missing_item_idx,rows_missing_either_idx,pct_missing_either_idx
0,implicit_valid,169855,48346,43718,82876,48.7922
1,implicit_test,169856,115885,43126,134128,78.9657


## 13. Create compact split summary tables

This section creates reporting tables covering:
- split sizes
- user and recipe counts
- date ranges
- unseen mapping behaviour

In [34]:
explicit_split_summary = pd.DataFrame([
    split_summary(explicit_train_mapped, "explicit_train"),
    split_summary(explicit_valid_mapped, "explicit_valid"),
    split_summary(explicit_test_mapped, "explicit_test"),
])

implicit_split_summary = pd.DataFrame([
    split_summary(implicit_train_mapped, "implicit_train"),
    split_summary(implicit_valid_mapped, "implicit_valid"),
    split_summary(implicit_test_mapped, "implicit_test"),
])

joined_split_summary = pd.DataFrame([
    split_summary(joined_train, "joined_train"),
    split_summary(joined_valid, "joined_valid"),
    split_summary(joined_test, "joined_test"),
])

display(explicit_split_summary)
display(implicit_split_summary)
display(joined_split_summary)

,split,rows,users,recipes,min_date,max_date
0,explicit_train,750064,97451,179915,2000-01-25 00:00:00,2010-04-24 00:00:00
1,explicit_valid,160728,37816,76045,2010-04-24 00:00:00,2012-09-09 00:00:00
2,explicit_test,160728,82968,61206,2012-09-09 00:00:00,2018-12-20 00:00:00


,split,rows,users,recipes,min_date,max_date
0,implicit_train,792656,106304,186564,2000-01-25 00:00:00,2010-06-17 00:00:00
1,implicit_valid,169855,46047,78532,2010-06-17 00:00:00,2012-12-28 00:00:00
2,implicit_test,169856,97133,61095,2012-12-28 00:00:00,2018-12-20 00:00:00


,split,rows,users,recipes,min_date,max_date
0,joined_train,792656,106304,186564,2000-01-25 00:00:00,2010-06-17 00:00:00
1,joined_valid,169855,46047,78532,2010-06-17 00:00:00,2012-12-28 00:00:00
2,joined_test,169856,97133,61095,2012-12-28 00:00:00,2018-12-20 00:00:00


## 14. Save mapping tables

The mapping tables are saved separately so that downstream models can reuse stable train-fitted encodings.

In [35]:
explicit_user_map.to_csv(MAPPINGS_DIR / "05_explicit_user_id_map.csv", index=False)
explicit_recipe_map.to_csv(MAPPINGS_DIR / "05_explicit_recipe_id_map.csv", index=False)

implicit_user_map.to_csv(MAPPINGS_DIR / "05_implicit_user_id_map.csv", index=False)
implicit_recipe_map.to_csv(MAPPINGS_DIR / "05_implicit_recipe_id_map.csv", index=False)

print("Mapping tables saved to:", MAPPINGS_DIR)

Mapping tables saved to: E:\UWE\Class Notes\Year 3\Digital Systems Project\Project V2\project\data\mappings


## 15. Save split datasets

Each split is saved as a separate parquet file for later experiments.

In [36]:
explicit_train_mapped.to_parquet(SPLITS_DIR / "explicit_train.parquet", index=False)
explicit_valid_mapped.to_parquet(SPLITS_DIR / "explicit_valid.parquet", index=False)
explicit_test_mapped.to_parquet(SPLITS_DIR / "explicit_test.parquet", index=False)

implicit_train_mapped.to_parquet(SPLITS_DIR / "implicit_train.parquet", index=False)
implicit_valid_mapped.to_parquet(SPLITS_DIR / "implicit_valid.parquet", index=False)
implicit_test_mapped.to_parquet(SPLITS_DIR / "implicit_test.parquet", index=False)

joined_train.to_parquet(SPLITS_DIR / "joined_train.parquet", index=False)
joined_valid.to_parquet(SPLITS_DIR / "joined_valid.parquet", index=False)
joined_test.to_parquet(SPLITS_DIR / "joined_test.parquet", index=False)

print("Split datasets saved to:", SPLITS_DIR)

Split datasets saved to: E:\UWE\Class Notes\Year 3\Digital Systems Project\Project V2\project\data\splits


## 16. Save reporting tables

In [37]:
explicit_split_summary.to_csv(TABLES_DIR / "05_explicit_split_summary.csv", index=False)
implicit_split_summary.to_csv(TABLES_DIR / "05_implicit_split_summary.csv", index=False)
joined_split_summary.to_csv(TABLES_DIR / "05_joined_split_summary.csv", index=False)

explicit_unseen_summary.to_csv(TABLES_DIR / "05_explicit_unseen_mapping_summary.csv", index=False)
implicit_unseen_summary.to_csv(TABLES_DIR / "05_implicit_unseen_mapping_summary.csv", index=False)

print("Split summary tables saved to:", TABLES_DIR)

Split summary tables saved to: E:\UWE\Class Notes\Year 3\Digital Systems Project\Project V2\project\outputs\tables


## 17. Save a compact JSON log

A compact log is saved for later write-up use and pipeline debugging.

In [38]:
split_log = {
    "train_fraction": TRAIN_FRAC,
    "valid_fraction": VALID_FRAC,
    "test_fraction": TEST_FRAC,
    "explicit_rows_total": int(len(explicit_model_df)),
    "implicit_rows_total": int(len(implicit_model_df)),
    "joined_rows_total": int(len(joined_model_df)),
    "explicit_train_rows": int(len(explicit_train_mapped)),
    "explicit_valid_rows": int(len(explicit_valid_mapped)),
    "explicit_test_rows": int(len(explicit_test_mapped)),
    "implicit_train_rows": int(len(implicit_train_mapped)),
    "implicit_valid_rows": int(len(implicit_valid_mapped)),
    "implicit_test_rows": int(len(implicit_test_mapped)),
    "joined_train_rows": int(len(joined_train)),
    "joined_valid_rows": int(len(joined_valid)),
    "joined_test_rows": int(len(joined_test)),
    "explicit_train_users": int(explicit_train["user_id"].nunique()),
    "explicit_train_recipes": int(explicit_train["recipe_id"].nunique()),
    "implicit_train_users": int(implicit_train["user_id"].nunique()),
    "implicit_train_recipes": int(implicit_train["recipe_id"].nunique()),
}

with open(LOGS_DIR / "05_split_log.json", "w", encoding="utf-8") as f:
    json.dump(split_log, f, indent=2)

print("Split log saved to:", LOGS_DIR / "05_split_log.json")

Split log saved to: E:\UWE\Class Notes\Year 3\Digital Systems Project\Project V2\project\outputs\logs\05_split_log.json


## 18. Summary

This notebook created chronological train, validation, and test splits for the explicit, implicit, and joined modelling datasets.

Key preprocessing decisions in this phase:
- rows were sorted chronologically before splitting
- global time-aware splits were used with a 70 / 15 / 15 partition
- user and item encoders were fitted using train data only
- validation and test were mapped using train-fitted encoders
- unseen users and items outside train were quantified rather than silently dropped
- split datasets and mapping tables were saved for downstream modelling

These outputs are now ready for:
- collaborative filtering
- matrix factorisation
- later feature engineering
- evaluation without temporal leakage